# Experiment 4 - Stage 1: Visual Expert Pre-training

**Objective**: Train a ResNet-18 to act as a robust Visual Feature Extractor (`visual_encoder.pth`).
**Method**: 
- Input: Chest X-Rays (CheXchonet)
- Augmentation: Aggressive (RandomAffine, ColorJitter) to handle noise/artifacts.
- Task: Binary Classification (30-day Readmission) for alignment with final task.
- Output: `visual_encoder.pth`

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import pandas as pd
import numpy as np
from tqdm import tqdm
import time

# Configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
EPOCHS = 5
LR = 1e-4
DATA_PATH = r"e:\平时作业\人工智能\data\master_index.csv"

# Check device
print(f"Using device: {DEVICE}")

In [ ]:
class VisualDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row['image_path']
        label = float(row['label'])
        
        image = torch.zeros(3, 224, 224)
        if pd.notna(img_path) and os.path.exists(img_path):
            try:
                img = Image.open(img_path).convert('RGB')
                if self.transform:
                    image = self.transform(img)
            except Exception as e:
                pass # Return zeros if corrupted
        
        return image, torch.tensor(label, dtype=torch.float32)

# Load Data
df_master = pd.read_csv(DATA_PATH)
df_valid_img = df_master[df_master['has_image'] == True].reset_index(drop=True)
print(f"Total samples with images: {len(df_valid_img)}")

# Transforms (Aggressive Augmentation as requested)
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomRotation(15),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Split
train_size = int(0.8 * len(df_valid_img))
val_size = len(df_valid_img) - train_size
train_df = df_valid_img.iloc[:train_size]
val_df = df_valid_img.iloc[train_size:]

train_dataset = VisualDataset(train_df, transform=train_transform)
val_dataset = VisualDataset(val_df, transform=train_transform) # Typically less aug for val, but sticking to simple

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
print("Data Loaders Ready")

In [ ]:
class VisualEngine(nn.Module):
    def __init__(self):
        super().__init__()
        # Load Pretrained ResNet18
        self.model = models.resnet18(pretrained=True)
        # Modify FC for feature extraction (512-dim output instead of logits)
        # But for Stage 1 Training, we need a refined classifier
        self.feature_extractor = nn.Sequential(*list(self.model.children())[:-1])
        
        # Temporary Classifier for Stage 1 Training
        self.classifier = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        features = self.feature_extractor(x)
        features = features.view(features.size(0), -1) # Flatten [B, 512, 1, 1] -> [B, 512]
        logits = self.classifier(features)
        return logits, features

model = VisualEngine().to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

print("Starting Visual Expert Training...")
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        images, labels = images.to(DEVICE), labels.to(DEVICE).unsqueeze(1)
        
        optimizer.zero_grad()
        logits, _ = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    print(f"Epoch {epoch+1} Loss: {running_loss/len(train_loader):.4f}")

# Save the Encoder (Feature Extractor + FC, but mostly we care about the extractor)
torch.save(model.state_dict(), "visual_encoder.pth")
print("Visual Expert Saved to visual_encoder.pth")